# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to record sets, fields, and columns use their Croissant `@id`s for precision and reproducibility.

### Dataset Source
The Croissant schema for this dataset can be found at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is available
!pip install -q mlcroissant pandas matplotlib

## 1. Data Loading

Load dataset metadata using `mlcroissant`, and review its high-level description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access the metadata object – display the dataset information
metadata = dataset.metadata
print(f"Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Date Published: {metadata.datePublished}\n")
print(f"Version: {metadata.version}\n")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")

## 2. Data Overview

Explore the dataset structure: list available record sets (`cr:RecordSet`), their fields, and collect their `@id`s for later use. All entities are referenced using their `@id`s.

In [ ]:
# List all record sets by @id
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print(f"Number of record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"Record set @id: {getattr(rs, '@id', None)}")
        print(f"  Name: {getattr(rs, 'name', None)}")
        print(f"  Description: {getattr(rs, 'description', None)}")
        fields = getattr(rs, 'field', [])
        print(f"  Fields ({len(fields)}):")
        for f in fields:
            print(f"    Field @id: {getattr(f, '@id', None)}, name: {getattr(f, 'name', None)}, dataType: {getattr(f, 'dataType', None)}")
        print()

## 3. Data Extraction

Load data from all available record sets into DataFrames. We use record set and field `@id`s for explicit references in all extraction and analysis steps.

In [ ]:
# Gather all record set @id values
record_set_ids = []
if getattr(dataset.metadata, 'recordSet', None):
    record_set_ids = [getattr(rs, '@id', None) for rs in dataset.metadata.recordSet]

print(f"Found record sets: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        # Load all records for the given record set @id
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set @id '{record_set_id}'. List of columns (fields as @id):")
            print(df.columns.tolist())
            display(df.head(3))
        else:
            print(f"No records found for record set @id '{record_set_id}'.")
    except Exception as e:
        print(f"Failed to load records for record set @id '{record_set_id}': {str(e)}")

if not dataframes:
    print("No data extracted – your dataset may not define explicit record sets or they may be empty.")

## 4. Exploratory Data Analysis (EDA)

We now process the data: select a numeric field by its `@id`, filter, normalize, and group. All field and record set references use their Croissant `@id`.

_You may need to adjust the field @id's in the variables below depending on the available record sets and fields in this dataset._

In [ ]:
# Example: You need to set the following variables to available @id values from above
# If no record sets are defined, you may need to skip this cell
if dataframes:
    # Choose the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Attempt to detect a numeric field id, fallback to user-input
    # We'll check the field types via metadata if available
    numeric_field_id = None
    group_field_id = None
    
    for rs in dataset.metadata.recordSet:
        if getattr(rs, '@id', None) == record_set_id:
            for f in getattr(rs, 'field', []):
                dtype = getattr(f, 'dataType', None)
                # Looking for 'schema:Float' or 'schema:Integer' types
                if dtype in ['schema:Float', 'schema:Integer'] and not numeric_field_id:
                    numeric_field_id = getattr(f, '@id', None)
                if dtype == 'schema:Text' and not group_field_id:
                    group_field_id = getattr(f, '@id', None)
            break
    
    print(f"Selected record set @id: {record_set_id}")
    print(f"Chosen numeric field @id: {numeric_field_id}")
    print(f"Chosen group (categorical) field @id: {group_field_id}\n")

    if numeric_field_id and numeric_field_id in df.columns:
        # Convert to numeric, handle missing values if any
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (mean): {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalize
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, field_norm]].head())

        # Group by selected category field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print(f"Group field {group_field_id} not found in DataFrame columns.")
    else:
        print(f"No suitable numeric field found for EDA in record set {record_set_id}.")
else:
    print("No data frames available to process.")

## 5. Visualization

Visualize the distribution of a chosen numeric field, or explore the relationship between a numeric and categorical field (if available) using Matplotlib and Pandas plotting.

In [ ]:
if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Bar plot if group field available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        grouped.plot(kind='bar', color='skyblue')
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available or no data loaded for visualization.")

## 6. Conclusion

In this notebook, you explored the FAIR² dataset via its Croissant schema using the `mlcroissant` library. 

- Dataset entities and data were referenced by their Croissant `@id` for reproducibility and clarity.
- Data loading, programmatic schema inspection, EDA filtering, normalization, grouping, and basic visualization were demonstrated.

**Note:**
- Some datasets may not expose explicit record sets or fields in their schema; this notebook is general and ready to adapt to other Croissant-compliant datasets.
- Always verify which record sets and fields (by `@id`) are available before proceeding with extraction or analysis.